This notebook runs the training for finetuning dinov2 on the image dataset

Please modify the `run` variable and the model [configuration](#configuration)

# Setup

In [ ]:
import os

import pandas as pd
import numpy as np

from tensorflow.keras.utils import image_dataset_from_directory
from tf.data import AUTOTUNE
from tf.config import list_physical_devices

os.environ["KERAS_BACKEND"] = "tensorflow" # Making sure keras backend is the right one
import keras
import keras_hub

In [ ]:
# Try to use requirements.txt and fallback on the full pip command.
!pip install -r requirements.txt || pip install pytest pylint ipdb jupyterlab numpy pandas matplotlib seaborn scikit-learn tensorflow timm transformers keras_hub==0.26.0

In [ ]:
# How do you want to run it?
run = "colab"
# run = "local"

In [ ]:
if run == "colab":
    # Connect to google drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Set the default root path of Kinoko project
    ROOT = "/content/drive/MyDrive/Colab Notebooks/lewagon/Kinoko"

    print(list_physical_devices('GPU'))
elif run == "local":
    ROOT = "../"
else:
    print("Error: variable `run` is set as an unknown type")


In [ ]:
# Setting image dataset directory
image_data_dir = f"{ROOT}/data/image_dataset"
image_dataset = image_dataset_from_directory(image_data_dir,
                                             labels="inferred",
                                             label_mode="binary",
                                             )

# Modeling

## Configuration

Modify this if you need 

In [ ]:
IMG_SIZE = 518 # DINOv2 Base preset expects 518x518
BATCH_SIZE = 32
EPOCHS = 50
SEED = 42

## Data preparation

In [ ]:
# train set is 70%
# val set is 15%
# test set is 15%

train_ds = image_dataset_from_directory(
    image_data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), # Changed to 518
    batch_size=BATCH_SIZE
)

test_val_ds = image_dataset_from_directory(
    image_data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), # Changed to 518
    batch_size=BATCH_SIZE
)

# Split test and val data
half_test_val_size = int(len(test_val_ds)/2)
test_ds = test_val_ds.take(half_test_val_size)
val_ds = test_val_ds.skip(half_test_val_size)

# Prefetching for faster loading in GPU
train_ds.prefetch(AUTOTUNE)
test_ds.prefetch(AUTOTUNE)
val_ds.prefetch(AUTOTUNE)

## Model loading

In [ ]:
backbone = keras_hub.models.DINOV2Backbone.from_preset("dinov2_base")

# Freeze the backbone
backbone.trainable = False

# Build the model
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# ImageNet Normalization
x = keras.layers.Rescaling(1/255.0)(inputs)
x = keras.layers.Normalization(
    mean=[0.485, 0.456, 0.406],
    variance=[0.229**2, 0.224**2, 0.225**2]
)(x)

# Pass to backbone
backbone_out = backbone({"images": x})  # convert to dict
outputs = backbone_out[:, 0, :]  # Get only CLS token

## New head

In [ ]:
x = keras.layers.Dense(512, activation="relu")(outputs)
x = keras.layers.Dropout(0.3)(x)
predictions = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=inputs, outputs=predictions)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", "recall", "precision"]
)

model.summary()

# Training

## Callbacks

In [ ]:
# --- CALLBACKS ---
callbacks = [
    # Stop early if val_loss stops improving
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    # Reduce LR on plateau (helps squeeze out last gains)
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7
    ),

    # Saving model if it's better than the one saved before
    keras.callbacks.ModelCheckpoint(
      filepath=f"{ROOT}/checkpoints/dinov2.keras",
      save_best_only=True,
      save_freq="epoch",  # every N epochs, or "epoch" for every one
    ),
]

## Training

In [ ]:
print("\nStarting training...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## Full fine tuning (optional) (not tested)

In [ ]:
# # --- 5. OPTIONAL: FULL FINE-TUNING ---
# # After the head is trained, you can unfreeze and train with a tiny learning rate
# print("\nUnfreezing backbone for fine-tuning...")
# backbone.trainable = True
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-6), # Extremely low LR
#     loss="categorical_crossentropy",
#     metrics=["accuracy"]
# )
# # model.fit(train_ds, validation_data=val_ds, epochs=2)

# # --- 6. EVALUATION & PREDICTION ---
# print("\nEvaluating model...")
# loss, accuracy = model.evaluate(val_ds)
# print(f"Validation Accuracy: {accuracy*100:.2f}%")

# # Prediction example
# sample_img = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype("float32")
# prediction = model.predict(sample_img)
# print(f"Prediction shape: {prediction.shape}")